<a href="https://colab.research.google.com/github/vikassinngh123/AI-ML-Learning/blob/main/06-Deep-Learning/01-PyTorch/01-PyTorch-Basic-Models/03_number_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
#For plots and visualization
import matplotlib.pyplot as plt
import seaborn as sns
#For data loading
from tensorflow.keras.datasets import mnist

from sklearn.model_selection import train_test_split

import torch
from torch import nn

In [2]:
(X,y),(_,_) = mnist.load_data()

In [3]:
X.shape,y.shape

((60000, 28, 28), (60000,))

In [4]:
X=X.reshape(60000,784)

In [5]:
X=X/255     #Dividing it by 255 gives values between 0-1(why we need values b/w 0-1 as if values are to high and lr is also high due to which the model make massive , violent updates to its weights during the first few epochs. which makes weights and the results go to negative and then ReLU converts it to 0 and gradient becomes zero )

In [6]:
X_train , X_test , y_train , y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [7]:
#Device agnostice code
device="cuda" if torch.cuda.is_available() else "cpu"

In [8]:
X_train=torch.tensor(X_train,dtype=torch.float32)
X_test=torch.tensor(X_test,dtype=torch.float32)
y_train=torch.tensor(y_train,dtype=torch.long)
y_test=torch.tensor(y_test,dtype=torch.long)

In [9]:
X_train,X_test=X_train.to(device),X_test.to(device)
y_train,y_test=y_train.to(device),y_test.to(device)

In [10]:
class numbermodel_ReLU(nn.Module):
  def __init__(self,input_features,hidden_features,output_features):
    '''Args:
            input_features(int):Number of input features
            hidden_features(int):Number of hidden features
            output_features(int):Number of output features
    '''
    super().__init__()
    self.sequential=nn.Sequential(
                                  nn.Linear(input_features,hidden_features),
                                  nn.ReLU(),
                                  nn.Linear(hidden_features,hidden_features),
                                  nn.ReLU(),
                                  nn.Linear(hidden_features,hidden_features),
                                  nn.ReLU(),
                                  nn.Linear(hidden_features,output_features)
                                 )

  def forward(self,x):
    return self.sequential(x)

model=numbermodel_ReLU(784,128,10).to(device)
model

numbermodel_ReLU(
  (sequential): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=128, bias=True)
    (5): ReLU()
    (6): Linear(in_features=128, out_features=10, bias=True)
  )
)

In [11]:
loss_fn=nn.CrossEntropyLoss()

optimizer=torch.optim.Adam(params=model.parameters(),lr=0.001)

def accuracy_fn(y_true,y_pred):
  correct=torch.eq(y_true,y_pred).sum().item()
  acc=correct/len(y_pred)
  return acc*100

In [12]:
epochs=1000

for epochs in range(epochs):
  model.train()

  y_logits=model(X_train)
  y_pred=torch.argmax(torch.softmax(y_logits,dim=1),dim=1)

  loss=loss_fn(y_logits,y_train)
  train_accuracy=accuracy_fn(y_train,y_pred)

  optimizer.zero_grad()

  loss.backward()

  optimizer.step()

  model.eval()
  with torch.inference_mode():
    test_logits=model(X_test)
    test_pred=torch.argmax(torch.softmax(test_logits,dim=1),dim=1)

    test_loss=loss_fn(test_logits,y_test)
    test_accuracy=accuracy_fn(y_test,test_pred)

  if epochs%100==0 or epochs==999:
    print(f"Epochs:{epochs} | Train_Loss:{loss: .2f} | Train_Accuracy:{train_accuracy: .2f} | Test_Loss:{test_loss: .2f} | Test_Accuracy:{test_accuracy: .2f}")

Epochs:0 | Train_Loss: 2.30 | Train_Accuracy: 10.78 | Test_Loss: 2.29 | Test_Accuracy: 16.93
Epochs:100 | Train_Loss: 0.19 | Train_Accuracy: 94.50 | Test_Loss: 0.20 | Test_Accuracy: 94.28
Epochs:200 | Train_Loss: 0.08 | Train_Accuracy: 97.70 | Test_Loss: 0.12 | Test_Accuracy: 96.54
Epochs:300 | Train_Loss: 0.03 | Train_Accuracy: 99.24 | Test_Loss: 0.10 | Test_Accuracy: 97.17
Epochs:400 | Train_Loss: 0.01 | Train_Accuracy: 99.85 | Test_Loss: 0.11 | Test_Accuracy: 97.24
Epochs:500 | Train_Loss: 0.00 | Train_Accuracy: 99.98 | Test_Loss: 0.13 | Test_Accuracy: 97.26
Epochs:600 | Train_Loss: 0.00 | Train_Accuracy: 100.00 | Test_Loss: 0.14 | Test_Accuracy: 97.19
Epochs:700 | Train_Loss: 0.00 | Train_Accuracy: 100.00 | Test_Loss: 0.15 | Test_Accuracy: 97.19
Epochs:800 | Train_Loss: 0.00 | Train_Accuracy: 100.00 | Test_Loss: 0.16 | Test_Accuracy: 97.20
Epochs:900 | Train_Loss: 0.00 | Train_Accuracy: 100.00 | Test_Loss: 0.17 | Test_Accuracy: 97.18
Epochs:999 | Train_Loss: 0.00 | Train_Accuracy: 

In [13]:
test_pred[:10],y_test[:10]

(tensor([7, 3, 8, 9, 3, 9, 7, 7, 5, 4], device='cuda:0'),
 tensor([7, 3, 8, 9, 3, 9, 7, 7, 5, 4], device='cuda:0'))

In [14]:
#Saving the models state dict
torch.save(model.state_dict(), "03_mnist_model_weights.pth")